# Ensemble Learning
This notebook provides an overview of ensemble learning, focusing on two specific techniques: stacking and blending.

Ensemble learning is a machine learning paradigm where multiple models (often called "base learners" or "level-0 models") are combined to solve a particular computational intelligence problem. The main idea is that by combining the predictions of several models, the overall performance and robustness of the system can be improved compared to using a single model.

The notebook demonstrates two ways to combine these models:

*   **Stacking:** This technique involves training a "meta-learner" (or "level-1 model") on the predictions of the base models. The notebook's first example of stacking uses cross-validation to generate out-of-fold predictions from the base models on the training data. These out-of-fold predictions then serve as the input features for the meta-learner.

*   **Blending:** A simpler variation of stacking, blending uses a separate hold-out validation set to generate the predictions from the base models. These predictions on the validation set are then used to train the meta-learner. The notebook's second example of stacking, which includes the original features along with the predictions, is a form of blending.

In essence, the notebook illustrates how combining different models through stacking and blending can potentially lead to improved predictive performance.

### Ignoring Convergence Warning

This cell imports the `warnings` module and `ConvergenceWarning` from scikit-learn and sets up a filter to ignore `ConvergenceWarning` messages. This is often done to prevent these warnings from cluttering the output, especially during iterative model training.

- `import warnings`: Imports the Python `warnings` module.
- `from sklearn.exceptions import ConvergenceWarning`: Imports the specific `ConvergenceWarning` class from scikit-learn.
- `warnings.filterwarnings("ignore", category=ConvergenceWarning)`: Configures the warnings module to ignore warnings of the category `ConvergenceWarning`. The first argument `"ignore"` specifies the action (ignore the warning), and the second argument `category=ConvergenceWarning` specifies the type of warning to ignore.
- The markdown then suggests re-executing the cell with the `LogisticRegression` model to see that the warning is now suppressed.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

### Load the Wine dataset

This cell loads the Wine dataset from scikit-learn, which is a classic dataset for classification tasks.

- `from sklearn.datasets import load_wine`: Imports the function to load the wine dataset.
- `import pandas as pd`: Imports the pandas library, which is commonly used for data manipulation and analysis.
- `import sklearn.tree as tree`: Imports the Decision Tree classifier from scikit-learn.
- `wine = load_wine()`: Loads the Wine dataset into the `wine` variable.
- `X = wine.data`: Assigns the feature data (the measurements of the wines) to the variable `X`.
- `y = wine.target`: Assigns the target variable (the class of each wine, which is either 0, 1, or 2) to the variable `y`.
- `feature_names = wine.feature_names`: Retrieves the names of the features.
- `target_names = wine.target_names`: Retrieves the names of the target classes.
- `print(...)`: Prints the shape of the data and target arrays, as well as the feature and target names, to give an initial overview of the dataset.

**Output Explanation:**

The output shows the dimensions of the data and target arrays, confirming that there are 178 samples with 13 features and a single target value for each sample. It also lists the names of the features and the three target classes.

In [1]:
from sklearn.datasets import load_wine
import pandas as pd
import sklearn.tree as tree

# Load the Wine dataset
wine = load_wine()

# Accessing the data and target
X = wine.data
y = wine.target

# Accessing feature and target names
feature_names = wine.feature_names
target_names = wine.target_names

print(f"Shape of data: {X.shape}")
print(f"Shape of target: {y.shape}")
print(f"Feature names: {feature_names}")
print(f"Target names: {target_names}")

Shape of data: (178, 13)
Shape of target: (178,)
Feature names: ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']
Target names: ['class_0' 'class_1' 'class_2']


## Stacking Ensemble Learning

Stacking (Stacked Generalization) is an ensemble machine learning method that combines the predictions of multiple base models (often called "level-0 models" or "base learners") using another model (called a "level-1 model" or "meta-learner"). The meta-learner is trained on the predictions of the base learners.

**How it works:**

1.  **Training Base Models:** The base models are trained on the original training dataset.
2.  **Generating Predictions:** Each base model makes predictions on the training dataset and/or a separate hold-out validation set.
3.  **Creating Meta-features:** The predictions from the base models become the new features for training the meta-learner. If a hold-out set is used, the base models are trained on the training data and predict on the hold-out data to create the meta-features for the meta-learner, which is then trained on these meta-features and the hold-out targets. Cross-validation can also be used to generate out-of-fold predictions for the entire training set to create the meta-features.
4.  **Training the Meta-learner:** The meta-learner is trained on these generated meta-features (the predictions from the base models) and the actual target values from the training or hold-out set.
5.  **Making Final Predictions:** When making predictions on new, unseen data, each base model first makes its prediction. These predictions are then fed as input to the trained meta-learner, which makes the final prediction.

**Advantages:**

-   Can potentially achieve higher accuracy than any single base model.
-   Combines the strengths of different types of models.

**Disadvantages:**

-   More complex to implement than simple averaging or voting ensembles.
-   Computationally more expensive due to training multiple models.
-   Risk of overfitting if the base models or meta-learner are not carefully chosen or tuned.

**Typical Base Learners:** Can be diverse models like Decision Trees, Logistic Regression, Support Vector Machines, Neural Networks, etc.

**Typical Meta-Learner:** Often a simpler model like Logistic Regression, Ridge Regression, or even another machine learning algorithm. The meta-learner learns how to optimally combine the predictions of the base learners.

### Splitting Data into Training and Testing Sets

This cell splits the loaded Wine dataset into training and testing sets. This is a standard practice in machine learning to evaluate the performance of a model on unseen data.

- `from sklearn.model_selection import train_test_split`: Imports the necessary function for splitting data.
- `X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)`: This line performs the split.
    - `X` and `y`: The feature data and target variable to be split.
    - `test_size=0.3`: Specifies that 30% of the data will be allocated to the testing set, and the remaining 70% to the training set.
    - `random_state=42`:  Ensures that the split is the same every time the code is run, making the results reproducible.
    - The function returns four arrays: `X_train` (training features), `X_test` (testing features), `y_train` (training targets), and `y_test` (testing targets).
- `print(...)`: These lines print the shapes of the resulting arrays, confirming the number of samples and features in each set.

**Output Explanation:**

The output shows the shapes of the training and testing sets. You can see that `X_train` and `y_train` contain 124 samples (70% of 178, rounded), while `X_test` and `y_test` contain 54 samples (30% of 178, rounded). The number of features (13) remains the same for `X_train` and `X_test`.

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")


Shape of X_train: (124, 13)
Shape of X_test: (54, 13)
Shape of y_train: (124,)
Shape of y_test: (54,)


### Stacking Function

This cell defines a Python function called `Stacking` which implements a stacking ensemble method. Stacking involves training multiple base models and then training a meta-model (in this case, a logistic regression model later) on the predictions of the base models. This function uses Stratified K-Fold cross-validation to generate out-of-fold predictions for the training data, which are then used to train the meta-model.

- `from sklearn.model_selection import StratifiedKFold`: Imports the function for stratified k-fold cross-validation.
- `import numpy as np`: Imports the NumPy library for numerical operations.
- `import pandas as pd`: Imports the pandas library.
- `def Stacking(model,train,y,test,n_fold):`: Defines the `Stacking` function which takes a base `model`, training data `train`, target variable `y`, testing data `test`, and number of folds `n_fold` as input.
- `folds=StratifiedKFold(n_splits=n_fold,random_state=1, shuffle=True)`: Initializes Stratified K-Fold cross-validation with the specified number of splits, a random state for reproducibility, and shuffling of the data.
- `test_pred=np.empty((test.shape[0],1),float)`: Initializes an empty NumPy array to store the predictions on the test set.
- `train_pred=np.empty((0,1),float)`: Initializes an empty NumPy array to store the out-of-fold predictions on the training set.
- The code then checks if the target variable `y`, training data `train`, and testing data `test` are pandas DataFrames and extracts their values or slices them accordingly.
- The `for` loop iterates through the folds generated by `StratifiedKFold`.
    - `train_indices,val_indices = folds.split(train,y_values)`: Gets the indices for the training and validation sets for the current fold.
    - `x_train,x_val=train.iloc[train_indices],train.iloc[val_indices]` or `x_train,x_val=train[train_indices],train[val_indices]`: Splits the training data into training and validation sets for the current fold.
    - `y_train_fold,y_val=y_values[train_indices],y_values[val_indices]`: Splits the target variable into training and validation sets for the current fold.
    - `model.fit(X=x_train,y=y_train_fold)`: Trains the base model on the training data for the current fold.
    - `train_pred=np.append(train_pred,model.predict(x_val))`: Makes predictions on the validation set for the current fold and appends them to `train_pred`.
- `test_pred = model.predict(test).reshape(-1, 1)`: After the loop, the base model is trained on the entire training set and makes predictions on the test set. These predictions are stored in `test_pred`.
- `return test_pred,train_pred.reshape(-1, 1)`: The function returns the predictions on the test set and the out-of-fold predictions on the training set.

In [11]:
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd

def Stacking(model,train,y,test,n_fold):
   print("Shape of data", train.shape)
   folds=StratifiedKFold(n_splits=n_fold,random_state=1, shuffle=True)
   test_pred=np.empty((test.shape[0],1),float)
   train_pred=np.empty((0,1),float)

   # Check if y is a pandas Series or DataFrame
   if isinstance(y, (pd.Series, pd.DataFrame)):
       y_values = y.values
   else:
       y_values = y

   i = 0
   test_preds = []
   for train_indices,val_indices in folds.split(train,y_values):
      print(f"Fold - {i}. Train Indices - {train_indices}. Test Indices - {val_indices}")
      i = i + 1
      print(f"Shape of Fold data {train_indices.shape} - {val_indices.shape}")
      # Check if train and test are pandas DataFrames
      if isinstance(train, pd.DataFrame):
          x_train,x_val=train.iloc[train_indices],train.iloc[val_indices]
      else:
          x_train,x_val=train[train_indices],train[val_indices]

      if isinstance(test, pd.DataFrame):
          # This line was incorrect as it was inside the train check
          pass # Placeholder, test is used later

      y_train_fold,y_val=y_values[train_indices],y_values[val_indices]

      model.fit(X=x_train,y=y_train_fold)

      # Predict on the validation split and append to the out-of-fold predictions
      pred = model.predict(x_val).reshape(-1, 1)
      train_pred = np.append(train_pred, pred, axis=0)

      test_preds.append(model.predict(test).reshape(-1, 1))

   test_pred = np.mean(test_preds, axis=0)


   return test_pred,train_pred.reshape(-1, 1)

### **What is Stratified Cross-Validation?**

**Stratified CV** is a variation where the data splits are made so that the distribution of the target classes is approximately the same in each fold as in the whole dataset.

For example:

* If 70% of the data is class 0 and 30% is class 1, stratified CV ensures that each fold also maintains this ratio.

This is important when:
✔ The dataset is **small**
✔ The classes are **imbalanced** (one class has far fewer samples than the other)

It prevents folds from being dominated by one class, which could bias the evaluation.

### ✅ Example: Stratified CV vs Regular CV

Let’s say you have 124 samples with a distribution like:

| Class | Count |
| ----- | ----- |
| 0     | 100   |
| 1     | 24    |

* In **regular CV**, some folds might randomly contain very few or even zero samples from class 1.
* In **stratified CV**, each fold will approximately have the same proportion → e.g., around 19 class 0 and 5 class 1 samples per fold.

### ✅ When should you prefer Stratified CV?

* ✅ Small datasets (like yours — 124 rows)
* ✅ Imbalanced datasets
* ✅ Classification problems (less relevant for regression tasks)

### Base Model 1: Decision Tree Classifier

This cell initializes a Decision Tree Classifier as the first base model and uses the `Stacking` function to generate out-of-fold predictions on the training data and predictions on the test data.

- `model1 = tree.DecisionTreeClassifier(random_state=1)`: Initializes a Decision Tree Classifier with a `random_state` for reproducibility.
- `test_pred1 ,train_pred1=Stacking(model=model1,n_fold=10, train=X_train,test=X_test,y=y_train)`: Calls the `Stacking` function with the Decision Tree model, 10 folds, the training data (`X_train`, `y_train`), and the test data (`X_test`). The function returns the test predictions (`test_pred1`) and the out-of-fold training predictions (`train_pred1`).
- `train_pred1=pd.DataFrame(train_pred1)`: Converts the out-of-fold training predictions to a pandas DataFrame.
- `test_pred1=pd.DataFrame(test_pred1)`: Converts the test predictions to a pandas DataFrame.

In [12]:
model1 = tree.DecisionTreeClassifier(random_state=1)

test_pred1, train_pred1 = Stacking(model=model1,n_fold=10, train=X_train,test=X_test,y=y_train)

train_pred1 = pd.DataFrame(train_pred1)
test_pred1 = pd.DataFrame(test_pred1)

Shape of data (124, 13)
Fold - 0. Train Indices - [  0   1   2   3   4   5   6   7   8  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  51  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  69  70  71  72  73  74  75
  76  77  78  79  80  81  82  83  84  85  87  88  90  91  92  93  94  98
  99 101 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118
 119 120 121]. Test Indices - [  9  50  52  68  86  89  95  96  97 100 102 122 123]
Shape of Fold data (111,) - (13,)
Fold - 1. Train Indices - [  0   1   2   3   4   5   6   7   8   9  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  28  29  30  31  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  52  53  54  55  56  57
  58  59  60  62  63  64  65  67  68  69  70  71  72  73  74  75  76  77
  78  79  80  82  84  85  86  88  89  90  91  92  93  94  95  96  97  99
 10

In [13]:
train_pred1.shape

(124, 1)

In [14]:
test_pred1.shape

(54, 1)

### Base Model 2: K-Nearest Neighbors Classifier

This cell initializes a K-Nearest Neighbors Classifier as the second base model and uses the `Stacking` function to generate out-of-fold predictions on the training data and predictions on the test data.

- `from sklearn.neighbors import KNeighborsClassifier`: Imports the K-Nearest Neighbors Classifier.
- `model2 = KNeighborsClassifier()`: Initializes a K-Nearest Neighbors Classifier with default parameters.
- `test_pred2 ,train_pred2=Stacking(model=model2,n_fold=10,train=X_train,test=X_test,y=y_train)`: Calls the `Stacking` function with the K-Nearest Neighbors model, 10 folds, the training data (`X_train`, `y_train`), and the test data (`X_test`). The function returns the test predictions (`test_pred2`) and the out-of-fold training predictions (`train_pred2`).
- `train_pred2=pd.DataFrame(train_pred2)`: Converts the out-of-fold training predictions to a pandas DataFrame.
- `test_pred2=pd.DataFrame(test_pred2)`: Converts the test predictions to a pandas DataFrame.

In [15]:
from sklearn.neighbors import KNeighborsClassifier

model2 = KNeighborsClassifier()

test_pred2, train_pred2 = Stacking(model=model2,n_fold=10,train=X_train,test=X_test,y=y_train)

train_pred2 = pd.DataFrame(train_pred2)
test_pred2 = pd.DataFrame(test_pred2)

Shape of data (124, 13)
Fold - 0. Train Indices - [  0   1   2   3   4   5   6   7   8  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  51  53  54  55  56
  57  58  59  60  61  62  63  64  65  66  67  69  70  71  72  73  74  75
  76  77  78  79  80  81  82  83  84  85  87  88  90  91  92  93  94  98
  99 101 103 104 105 106 107 108 109 110 111 112 113 114 115 116 117 118
 119 120 121]. Test Indices - [  9  50  52  68  86  89  95  96  97 100 102 122 123]
Shape of Fold data (111,) - (13,)
Fold - 1. Train Indices - [  0   1   2   3   4   5   6   7   8   9  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  28  29  30  31  33  34  35  36  37  38
  39  40  41  42  43  44  45  46  47  48  49  50  52  53  54  55  56  57
  58  59  60  62  63  64  65  67  68  69  70  71  72  73  74  75  76  77
  78  79  80  82  84  85  86  88  89  90  91  92  93  94  95  96  97  99
 10

### Meta-Model: Logistic Regression (First Stacking Implementation)

This cell combines the out-of-fold predictions from the two base models to create a new training set for the meta-model and the test predictions to create a new test set. It then trains a Logistic Regression model as the meta-model on these combined predictions and evaluates its performance on the test set.

- `from sklearn.linear_model import LogisticRegression`: Imports the Logistic Regression model.
- `df = pd.concat([train_pred1, train_pred2], axis=1)`: Concatenates the out-of-fold training predictions from the Decision Tree (`train_pred1`) and K-Nearest Neighbors (`train_pred2`) along the columns (axis=1) to create the training data for the meta-model.
- `df_test = pd.concat([test_pred1, test_pred2], axis=1)`: Concatenates the test predictions from the Decision Tree (`test_pred1`) and K-Nearest Neighbors (`test_pred2`) along the columns (axis=1) to create the test data for the meta-model.
- `model = LogisticRegression(random_state=1)`: Initializes a Logistic Regression model as the meta-model with a `random_state` for reproducibility.
- `model.fit(df,y_train)`: Trains the Logistic Regression meta-model on the combined out-of-fold training predictions (`df`) and the original training targets (`y_train`).
- `model.score(df_test, y_test)`: Evaluates the performance of the meta-model on the combined test predictions (`df_test`) and the original test targets (`y_test`) using the accuracy score.

**Output Explanation:**

The output shows the accuracy score of the Logistic Regression meta-model on the test set. A score of 0.388... indicates the proportion of correctly classified samples in the test set.

In [16]:
from sklearn.linear_model import LogisticRegression
df = pd.concat([train_pred1, train_pred2], axis=1)
df_test = pd.concat([test_pred1, test_pred2], axis=1)

model = LogisticRegression(random_state=1)
model.fit(df,y_train)
model.score(df_test, y_test)

0.3888888888888889

## Blending Ensemble Technique

Blending, also known as stacked generalization with a hold-out set, is an ensemble learning technique that combines the predictions of multiple individual models (base models or level-0 models) to make a final prediction using another model (a meta-model or level-1 model). It's a simpler variation of stacking.

Here's how blending typically works:

1.  **Splitting the Data:** The original dataset is split into three parts: a training set, a validation (or blend) set, and a test set.
2.  **Training Base Models:** The base models are trained independently on the *training set*.
3.  **Generating Predictions on the Blend Set:** Each trained base model makes predictions on the *validation set*. These predictions become the new features for the meta-model.
4.  **Training the Meta-Model:** A separate meta-model is trained on the predictions generated from the *validation set* by the base models. The target for the meta-model is the actual target variable from the *validation set*.
5.  **Generating Predictions on the Test Set:** The trained base models also make predictions on the *test set*.
6.  **Making Final Predictions:** The predictions from the base models on the *test set* are fed as input to the trained meta-model, which then makes the final prediction on the unseen data.

**Key Difference from Standard Stacking (as shown in the preceding code):**

The primary difference lies in how the training data for the meta-model is generated.

*   **Standard Stacking (like in the preceding code):** Uses cross-validation on the *entire training set* to generate "out-of-fold" predictions. The base models are trained on K-1 folds and predict on the remaining fold. This is repeated K times, and the out-of-fold predictions are combined to form the training set for the meta-model. This approach uses all the training data to generate meta-features for the meta-model.
*   **Blending:** Uses a dedicated *validation set* to generate the meta-features. The base models are trained *only* on the initial training set and then predict on the hold-out validation set. This is simpler to implement but uses less data to train both the base models (compared to standard stacking's cross-validation) and the meta-model.

**Advantages of Blending:**

*   **Simplicity:** It is conceptually simpler and easier to implement than standard stacking, especially when not using cross-validation for meta-feature generation.
*   **Faster Training:** Training is typically faster as cross-validation is not required for the base models to generate meta-features for the meta-model's training data.

**Disadvantages of Blending:**

*   **Less Data for Training:** Both the base models and the meta-model are trained on smaller subsets of the original data (training set for base models, validation set predictions for the meta-model) compared to standard stacking. This can lead to models that are less robust or don't capture the data distribution as well.
*   **Sensitivity to Validation Set Split:** The performance can be highly dependent on the specific split of the validation set. A poor split might not represent the data well, leading to a suboptimal meta-model.

In essence, blending is a streamlined version of stacking that sacrifices some data usage for simpler implementation and faster execution. The choice between standard stacking and blending often depends on the dataset size, complexity, and the computational resources available.

### Splitting Data into Three Parts (Train, Validation, Test)

This cell splits the original dataset (`X` and `y`) into three parts: a training set, a validation set, and a test set. This is an alternative splitting strategy compared to the previous train-test split and is often used in model development and evaluation, particularly in scenarios involving hyperparameter tuning where a separate validation set is needed.

- `X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)`: The data is first split into a training set (60%) and a temporary set (40%) using `train_test_split`.
- `X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42)`: The temporary set (`X_temp` and `y_temp`) is then split into a validation set (80% of the temporary set, which is 32% of the original data) and a test set (20% of the temporary set, which is 8% of the original data). The `test_size=0.2` here refers to the size relative to `X_temp` and `y_temp`.
- `print(...)`: These lines print the shapes of the resulting training, validation, and test sets, showing the number of samples and features in each.

**Output Explanation:**

The output confirms the shapes of the three resulting sets. You can see the number of samples in `X_train`, `X_val`, and `X_test`, and that the number of features (13) is consistent across the feature sets. The corresponding target sets (`y_train`, `y_val`, `y_test`) have the same number of samples as their respective feature sets.

In [23]:
# prompt: Split data X into 3 parts X_train, X_test and X_test  and y as well

# Splitting the original X and y into three parts
# We first split into train and the rest (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)

# Then split the rest into validation and test sets
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.2, random_state=42) # 0.5 of 0.4 is 0.2

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_val: {X_val.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_val: {y_val.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of X_train: (106, 13)
Shape of X_val: (57, 13)
Shape of X_test: (15, 13)
Shape of y_train: (106,)
Shape of y_val: (57,)
Shape of y_test: (15,)


### Base Models on Train/Validation/Test Split

This cell trains the two base models (Decision Tree and K-Nearest Neighbors) on the training data (`X_train`, `y_train`) from the three-part split and makes predictions on the validation set (`X_val`) and test set (`X_test`).

- `model1 = tree.DecisionTreeClassifier()`: Initializes a Decision Tree Classifier.
- `model1.fit(X_train, y_train)`: Trains the Decision Tree model on the training data.
- `val_pred1=model1.predict(X_val)`: Makes predictions on the validation set using the trained Decision Tree model.
- `test_pred1=model1.predict(X_test)`: Makes predictions on the test set using the trained Decision Tree model.
- `val_pred1=pd.DataFrame(val_pred1)`: Converts the validation predictions from the Decision Tree to a pandas DataFrame.
- `test_pred1=pd.DataFrame(test_pred1)`: Converts the test predictions from the Decision Tree to a pandas DataFrame.
- `model2 = KNeighborsClassifier()`: Initializes a K-Nearest Neighbors Classifier.
- `model2.fit(X_train,y_train)`: Trains the K-Nearest Neighbors model on the training data.
- `val_pred2=model2.predict(X_val)`: Makes predictions on the validation set using the trained K-Nearest Neighbors model.
- `test_pred2=model2.predict(X_test)`: Makes predictions on the test set using the trained K-Nearest Neighbors model.
- `val_pred2=pd.DataFrame(val_pred2)`: Converts the validation predictions from the K-Nearest Neighbors to a pandas DataFrame.
- `test_pred2=pd.DataFrame(test_pred2)`: Converts the test predictions from the K-Nearest Neighbors to a pandas DataFrame.

In [24]:
model1 = tree.DecisionTreeClassifier()
model1.fit(X_train, y_train)
val_pred1=model1.predict(X_val)
test_pred1=model1.predict(X_test)

val_pred1=pd.DataFrame(val_pred1)
test_pred1=pd.DataFrame(test_pred1)

model2 = KNeighborsClassifier()
model2.fit(X_train,y_train)
val_pred2=model2.predict(X_val)
test_pred2=model2.predict(X_test)

val_pred2=pd.DataFrame(val_pred2)
test_pred2=pd.DataFrame(test_pred2)

### Meta-Model: Logistic Regression (Second Stacking Implementation)

This cell concatenates the original validation data and the predictions from the base models on the validation set to create a new training set for the meta-model. Similarly, it concatenates the original test data and the predictions from the base models on the test set to create a new test set for the meta-model. It then trains a Logistic Regression model as the meta-model on the validation set and evaluates its performance on the test set. This is a slightly different approach to stacking where the original features are included alongside the base model predictions.

- `df_val=pd.concat([pd.DataFrame(X_val), val_pred1,val_pred2],axis=1)`: Concatenates the original validation features (`X_val` converted to a DataFrame), the validation predictions from the Decision Tree (`val_pred1`), and the validation predictions from the K-Nearest Neighbors (`val_pred2`) along the columns to create the training data for the meta-model.
- `df_test=pd.concat([pd.DataFrame(X_test), test_pred1,test_pred2],axis=1)`: Concatenates the original test features (`X_test` converted to a DataFrame), the test predictions from the Decision Tree (`test_pred1`), and the test predictions from the K-Nearest Neighbors (`test_pred2`) along the columns to create the test data for the meta-model.
- `model = LogisticRegression()`: Initializes a Logistic Regression model as the meta-model.
- `model.fit(df_val,y_val)`: Trains the Logistic Regression meta-model on the combined validation data (`df_val`) and the validation targets (`y_val`).
- `model.score(df_test,y_test)`: Evaluates the performance of the meta-model on the combined test data (`df_test`) and the test targets (`y_test`) using the accuracy score.

**Output Explanation:**

The output shows the accuracy score of the Logistic Regression meta-model on the test set using this stacking approach. A score of 0.933... indicates the proportion of correctly classified samples in the test set.

In [28]:
df_val=pd.concat([val_pred1,val_pred2],axis=1)
df_test=pd.concat([test_pred1,test_pred2],axis=1)
df_val1 = df_val

model = LogisticRegression()
model.fit(df_val,y_val)
model.score(df_test,y_test)

0.9333333333333333

In [27]:
df_val=pd.concat([pd.DataFrame(X_val), val_pred1,val_pred2],axis=1)
df_test=pd.concat([pd.DataFrame(X_test), test_pred1,test_pred2],axis=1)
df_val2 = df_val

model = LogisticRegression()
model.fit(df_val,y_val)
model.score(df_test,y_test)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


1.0

### Predict using the Meta-Model

This cell uses the trained Logistic Regression meta-model (`model`) to make predictions on the combined test data (`df_test`).

- `y_pred = model.predict(df_test)`: Makes predictions on the test data using the trained meta-model and stores the predicted class labels in the `y_pred` variable.

In [ ]:
y_pred = model.predict(df_test)

### Display Predictions

This cell displays the predicted class labels generated by the meta-model on the test set.

- `y_pred`: This line simply outputs the contents of the `y_pred` variable, which is a NumPy array containing the predicted class labels for each sample in the test set.

**Output Explanation:**

The output is a NumPy array showing the predicted class for each of the samples in the test set based on the meta-model's performance.

In [ ]:
y_pred

array([1, 1, 0, 0, 1, 1, 0, 0, 0, 2, 1, 1, 2, 0, 0])

### Display Validation DataFrame

This cell displays the first few rows of the `df_val` DataFrame, which was created by concatenating the original validation features and the validation predictions from the base models.

- `df_val`: This line outputs the contents of the `df_val` DataFrame. By default, Colab's `display` function will show the entire DataFrame if it's not too large, or a truncated version.

**Output Explanation:**

The output is a tabular representation of the `df_val` DataFrame. You can see the original features of the validation set (columns 0-12) followed by the predictions from the two base models on the validation set (columns with header 0, which are the concatenated prediction columns). This DataFrame was used to train the meta-model in the previous steps.

In [30]:
df_val2

,0,1,2,3,4,5,6,7,8,9,10,11,12,0,0
0,12.33,1.10,2.28,16.0,101.0,2.05,1.09,0.63,0.41,3.270000,1.25,1.67,680.0,1,2
1,12.33,0.99,1.95,14.8,136.0,1.90,1.85,0.35,2.76,3.400000,1.06,2.31,750.0,1,0
2,13.83,1.65,2.60,17.2,94.0,2.45,2.99,0.22,2.29,5.600000,1.24,3.37,1265.0,0,0
3,13.58,1.66,2.36,19.1,106.0,2.86,3.19,0.22,1.95,6.900000,1.09,2.88,1515.0,0,0
4,13.48,1.67,2.64,22.5,89.0,2.60,1.10,0.52,2.29,11.750000,0.57,1.78,620.0,2,1
5,13.71,1.86,2.36,16.6,101.0,2.61,2.88,0.27,1.69,3.800000,1.11,4.00,1035.0,1,0
6,11.41,0.74,2.50,21.0,88.0,2.48,2.01,0.42,1.44,3.080000,1.10,2.31,434.0,1,1
7,13.88,1.89,2.59,15.0,101.0,3.25,3.56,0.17,1.70,5.430000,0.88,3.56,1095.0,0,0
8,13.62,4.95,2.35,20.0,92.0,2.00,0.80,0.47,1.02,4.400000,0.91,2.05,550.0,2,1
9,13.68,1.83,2.36,17.2,104.0,2.42,2.69,0.42,1.97,3.840000,1.23,2.87,990.0,0,0


In [29]:
df_val1

,0,0
0,1,2
1,1,0
2,0,0
3,0,0
4,2,1
5,1,0
6,1,1
7,0,0
8,2,1
9,0,0


# Comparison

| Aspect            | Standard Stacking                           | Blending                         |
| ----------------- | ------------------------------------------- | -------------------------------- |
| Data use          | All training data used via cross-validation | Part of data used for validation |
| Robustness        | More robust, reduces overfitting            | Simpler but can overfit easily   |
| Complexity        | Requires K-fold setup                       | Easier to implement              |
| Meta-features     | Out-of-fold predictions for all samples     | Predictions only on hold-out set |
| Final performance | Usually better with more generalization     | Depends on validation set size   |
